# Insurance Claims Data Model — Exploration & SQL Execution Notebook

This notebook demonstrates how the data model works in practice by:
- Loading synthetic datasets  
- Creating an in‑memory SQLite database  
- Executing the physical SQL schema  
- Inserting sample data  
- Running analytical SQL queries  
- Displaying results  

This mirrors the workflow of a Junior Data Modeller validating a schema.

---

## 1. Setup

Import required libraries and create a connection to an in‑memory SQLite database.

In [3]:
import sqlite3
import pandas as pd

# Create in-memory SQLite database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

## 2. Load Synthetic Data

Load the CSV files from the `data/synthetic/` folder into pandas DataFrames.

In [4]:
customers = pd.read_csv("../data/customers.csv")
policies = pd.read_csv("../data/policies.csv")
claims = pd.read_csv("../data/claims.csv")
payments = pd.read_csv("../data/payments.csv")
assessors = pd.read_csv("../data/assessors.csv")
assessments = pd.read_csv("../data/assessments.csv")

customers.head()


,CustomerID,FirstName,LastName,DOB,Email,Phone
0,1,Shaggy,Rogers,1988-06-21,shaggy@mysteryinc.com,400123456
1,2,Velma,Dinkley,1992-03-12,velma@mysteryinc.com,400654321
2,3,Emperor,Kuzco,1500-01-01,kuzco@palace.gov,400789456
3,4,Lilo,Pelekai,2002-11-05,lilo@ohana.org,400555123
4,5,Wayne,McClane,1980-05-14,wayne.mcclane@nakatomi.com,400999888


## 3. Create Database Schema

Execute the SQL schema from `models/physical/schema.sql`.

In [5]:
with open("../models/physical/schema.sql", "r") as f:
    schema_sql = f.read()

cursor.executescript(schema_sql)
conn.commit()


## 4. Insert Synthetic Data into Tables

In [6]:
customers.to_sql("Customer", conn, if_exists="append", index=False)
policies.to_sql("Policy", conn, if_exists="append", index=False)
claims.to_sql("Claim", conn, if_exists="append", index=False)
payments.to_sql("Payment", conn, if_exists="append", index=False)
assessors.to_sql("Assessor", conn, if_exists="append", index=False)
assessments.to_sql("Assessment", conn, if_exists="append", index=False)


4

## 5. Run Sample SQL Queries

These queries demonstrate how the model supports reporting and analysis.

In [7]:
# 5.1 Claims per customer
query = """
SELECT 
    c.CustomerID,
    c.FirstName,
    c.LastName,
    COUNT(cl.ClaimID) AS TotalClaims
FROM Customer c
JOIN Policy p ON c.CustomerID = p.CustomerID
JOIN Claim cl ON p.PolicyID = cl.PolicyID
GROUP BY c.CustomerID, c.FirstName, c.LastName;
"""

pd.read_sql_query(query, conn)


,CustomerID,FirstName,LastName,TotalClaims
0,1,Shaggy,Rogers,1
1,2,Velma,Dinkley,1
2,3,Emperor,Kuzco,1
3,4,Lilo,Pelekai,1
4,5,Wayne,McClane,1


In [8]:
# 5.2 Total payments per claim
query = """
SELECT 
    ClaimID,
    SUM(PaymentAmount) AS TotalPaid
FROM Payment
GROUP BY ClaimID;
"""

pd.read_sql_query(query, conn)


,ClaimID,TotalPaid
0,1002,8000
1,1005,15000


In [9]:
# 5.3 Claims by status
query = """
SELECT 
    ClaimStatus,
    COUNT(*) AS Count
FROM Claim
GROUP BY ClaimStatus;
"""

pd.read_sql_query(query, conn)


,ClaimStatus,Count
0,Closed,2
1,Open,1
2,Pending,1
3,Rejected,1


In [10]:
# 5.4 Assessments per assessor
query = """
SELECT 
    a.AssessorID,
    a.Name,
    COUNT(asmt.AssessmentID) AS TotalAssessments
FROM Assessor a
LEFT JOIN Assessment asmt ON a.AssessorID = asmt.AssessorID
GROUP BY a.AssessorID, a.Name;
"""

pd.read_sql_query(query, conn)


,AssessorID,Name,TotalAssessments
0,9001,Emily Carter,2
1,9002,James Lee,1
2,9003,Miranda Priestly,1


## 6. Explore the Data Model Interactively

Use pandas to inspect joins and relationships visually.

In [11]:
merged = claims.merge(policies, on="PolicyID").merge(customers, on="CustomerID")
merged.head()

,ClaimID,PolicyID,ClaimDate,ClaimType,ClaimStatus,ClaimAmount,CustomerID,PolicyType,StartDate,EndDate,PremiumAmount,FirstName,LastName,DOB,Email,Phone
0,1001,101,2024-03-10,Collision,Open,2500.00,1,Auto,2024-01-01,2024-12-31,89.99,Shaggy,Rogers,1988-06-21,shaggy@mysteryinc.com,400123456
1,1002,102,2024-04-22,Fire,Closed,8000.00,2,Home,2024-02-15,2025-02-14,120.50,Velma,Dinkley,1992-03-12,velma@mysteryinc.com,400654321
2,1003,103,2024-05-05,MagicalIncident,Rejected,9999.99,3,Palace,2024-03-01,2025-03-01,500.00,Emperor,Kuzco,1500-01-01,kuzco@palace.gov,400789456
3,1004,104,2024-06-12,WaterDamage,Pending,1200.00,4,Home,2024-01-10,2025-01-10,75.25,Lilo,Pelekai,2002-11-05,lilo@ohana.org,400555123
4,1005,105,2024-07-01,Explosion,Closed,15000.00,5,Commercial,2024-04-01,2025-04-01,300.00,Wayne,McClane,1980-05-14,wayne.mcclane@nakatomi.com,400999888


## 7. Conclusion

This notebook validates the insurance claims data model by:
- Successfully loading structured data  
- Creating a relational schema  
- Enforcing relationships  
- Running analytical queries  
- Demonstrating real‑world usage  

This mirrors the workflow of a Junior Data Modeller working with business stakeholders and data engineers to ensure data structures support reporting and operational needs.